# Session Index and Trial Catalog for Monkey N

This notebook converts the NWB-level exploratory outputs into a clean, decoder-ready metadata layer.

## Objectives
1. Load the outputs generated by Notebook 01 (EDA).
2. Build a canonical session index.
3. Extract and standardize trial tables from all NWB sessions.
4. Summarize target-style composition, trial timing, and session-level consistency.
5. Export tables for later decoding and adaptive-threshold notebooks.

## Inputs
- `/kaggle/working/tables_eda/session_inventory.csv`
- `/kaggle/working/tables_eda/nwb_session_summary.csv`
- `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N/*.nwb`

## Outputs
- Session master table
- Trial catalog across all sessions
- Session-level trial summary
- QC figures for longitudinal and task-structure analysis

In [ ]:
!pip install pynwb h5py

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from pynwb import NWBHDF5IO

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.7)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

## Paths

This notebook reuses the exact Kaggle dataset path and the generated EDA outputs from Notebook 01.

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

EDA_TABLE_DIR = Path("/kaggle/working/tables_eda")
EDA_META_DIR = Path("/kaggle/working/meta_eda")

WORK_ROOT = Path("/kaggle/working")
OUT_TABLE_DIR = WORK_ROOT / "tables_session_index"
OUT_FIG_DIR = WORK_ROOT / "figures_session_index"
OUT_META_DIR = WORK_ROOT / "meta_session_index"

OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

SESSION_INVENTORY_CSV = EDA_TABLE_DIR / "session_inventory.csv"
SESSION_SUMMARY_CSV = EDA_TABLE_DIR / "nwb_session_summary.csv"

print("DATASET_DIR:", DATASET_DIR)
print("SESSION_INVENTORY_CSV:", SESSION_INVENTORY_CSV)
print("SESSION_SUMMARY_CSV:", SESSION_SUMMARY_CSV)

In [ ]:
assert DATASET_DIR.exists(), f"Missing dataset directory: {DATASET_DIR}"
assert SESSION_INVENTORY_CSV.exists(), f"Missing EDA file: {SESSION_INVENTORY_CSV}"
assert SESSION_SUMMARY_CSV.exists(), f"Missing EDA file: {SESSION_SUMMARY_CSV}"

inventory_df = pd.read_csv(SESSION_INVENTORY_CSV)
session_df = pd.read_csv(SESSION_SUMMARY_CSV)

inventory_df["session_date"] = pd.to_datetime(inventory_df["session_date"], errors="coerce")
session_df["session_date"] = pd.to_datetime(session_df["session_date"], errors="coerce")

print("inventory_df:", inventory_df.shape)
print("session_df:", session_df.shape)

display(inventory_df.head())
display(session_df.head())

In [ ]:
def extract_session_date_from_name(file_name):
    m = re.search(r"_ses-(\d{8})_", str(file_name))
    if m is None:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")

def infer_target_style(row):
    vals = []
    for col in ["session_description", "identifier", "file_name"]:
        val = row.get(col, None)
        if pd.notna(val):
            vals.append(str(val).lower())

    joined = " ".join(vals)

    if " target style co" in joined or "_co_" in joined or joined.endswith("_co_nwb"):
        return "CO"
    if " target style rd" in joined or "_rd_" in joined or joined.endswith("_rd_nwb"):
        return "RD"

    m = re.search(r"target style\s+([a-z0-9]+)", joined)
    if m:
        return m.group(1).upper()

    return "UNKNOWN"

def safe_read_trials(nwb_path):
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        if nwb.trials is None:
            return pd.DataFrame()
        return nwb.trials.to_dataframe().reset_index()